# Simulating the UCJ ansatz with fermionic backpropagation

This guide demonstrates how to simulate a one-layer [UCJ ansatz](../explanations/lucj.ipynb) using the algorithm described [here](https://arxiv.org/abs/2607.21337). This method can be used to efficiently simulate any one-layer UCJ ansatz exactly, with an optional final orbital rotation. It computes the expectation value of a one/two-body operator in the Heisenberg picture in $O(N^7)$ time for $N$ spatial orbitals.

In [1]:
import warnings
from collections import defaultdict

import pyscf
import pyscf.cc
import scipy

import ffsim
from ffsim.variational.ucj_energy import (
    ucj_energy_and_grad_func_spin_balanced,
    ucj_energy_and_grad_func_spin_unbalanced,
    ucj_energy_spin_balanced,
    ucj_energy_spin_unbalanced,
)

warnings.formatwarning = lambda msg, *args, **kwargs: f"Warning: {msg}\n"

## LUCJ ansatz for a closed-shell molecule
We'll construct the ansatz for a nitrogen molecule in the 6-31g basis set. Since it's a closed-shell system, use the spin-balanced UCJ ansatz. We will restrict the pair connectivity, however fermionic backpropagation in general will work for any connectivity.

In [ ]:
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="6-31g",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular data and Hamiltonian
scf = pyscf.scf.RHF(mol).run()
mol_data = ffsim.MolecularData.from_scf(scf, active_space=active_space)
norb, nelec = mol_data.norb, mol_data.nelec
mol_hamiltonian = mol_data.hamiltonian
print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()

n_reps = 1

# Define interactions
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None

# Use the backend implementable pairs_ab to construct the ucj_op
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    ccsd.t2,
    t1=ccsd.t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization.
    # Additionally, you may want to set the multi_stage_start or multi_stage_step
    # arguments (or both) to obtain a better result at increased computational cost.
    # See the API documentation for details.
    optimize=True,
    options=dict(maxiter=100),
)

Now, let's simulate the ansatz using the backpropagation method. 

In [ ]:
energy = ucj_energy_spin_balanced(ucj_op, mol_hamiltonian, nelec)
print(f"HF energy: {scf.e_tot:.6f}")
print(f"LUCJ Energy: {energy:.6f}")
print(f"CCSD energy: {ccsd.e_tot:.6f}")

We can also variationally optimize the ansatz parameters to achieve lower ground state energies. This is particularly useful for strongly correlated systems where the CCSD parameters may not be optimal. `ucj_energy_and_grad_func_spin_balanced` returns a callable that computes the energy and its gradient with respect to the ansatz parameters using JAX for autodifferentiation. We pass this callable directly to `scipy.optimize.minimize`. Note that for these examples, the number of terms is manageable on a GPU, but for larger systems, GPU acceleration might necessitate chunking the tensors, which can be set using the `chunk_size` argument.

In [ ]:
info = defaultdict(list)


def callback(intermediate_result: scipy.optimize.OptimizeResult):
    print(f" {intermediate_result.fun:.6f}")


value_and_grad = ucj_energy_and_grad_func_spin_balanced(
    ucj_op,
    mol_hamiltonian,
    nelec,
    interaction_pairs=(pairs_aa, pairs_ab),
    chunk_size=256,
)
result = scipy.optimize.minimize(
    value_and_grad,
    ucj_op.to_parameters(interaction_pairs=(pairs_aa, pairs_ab)),
    method="L-BFGS-B",
    jac=True,
    callback=callback,
    options=dict(maxiter=5),
)
optimal_ucj = ffsim.UCJOpSpinBalanced.from_parameters(
    result.x,
    norb=norb,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    with_final_orbital_rotation=ucj_op.final_orbital_rotation is not None,
)

## UCJ ansatz for an open-shell molecule
We'll use a hydroxyl radical in the 6-31g basis set as an example of an open-shell system. For this example, we'll use unrestricted pair connectivity, i.e. the full UCJ ansatz.

In [ ]:
# Build HO molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["H", (0, 0, 0)], ["O", (0, 0, 1.1)]],
    basis="6-31g",
    spin=1,
    symmetry="Coov",
)

# Get molecular data and Hamiltonian
scf = pyscf.scf.ROHF(mol).run()
mol_data = ffsim.MolecularData.from_scf(scf)
norb, nelec = mol_data.norb, mol_data.nelec
mol_hamiltonian = mol_data.hamiltonian
print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(scf).run()

# Use the backend implementable pairs_ab to construct the ucj_op
ucj_op = ffsim.UCJOpSpinUnbalanced.from_t_amplitudes(
    ccsd.t2,
    t1=ccsd.t1,
    n_reps=1,
    # Setting optimize=True enables the "compressed" factorization.
    # Additionally, you may want to set the multi_stage_start or multi_stage_step
    # arguments (or both) to obtain a better result at increased computational cost.
    # See the API documentation for details.
    optimize=True,
    options=dict(maxiter=100),
)

In [ ]:
energy = ucj_energy_spin_unbalanced(ucj_op, mol_hamiltonian, nelec)
print(f"HF energy: {scf.e_tot:.6f}")
print(f"UCJ Energy: {energy:.6f}")
print(f"CCSD energy: {ccsd.e_tot:.6f}")

In [ ]:
info = defaultdict(list)


def callback(intermediate_result: scipy.optimize.OptimizeResult):
    print(f" {intermediate_result.fun:.6f}")


value_and_grad = ucj_energy_and_grad_func_spin_unbalanced(
    ucj_op,
    mol_hamiltonian,
    nelec,
    chunk_size=256,
)
result = scipy.optimize.minimize(
    value_and_grad,
    ucj_op.to_parameters(),
    method="L-BFGS-B",
    jac=True,
    callback=callback,
    options=dict(maxiter=5),
)
optimal_ucj = ffsim.UCJOpSpinUnbalanced.from_parameters(
    result.x,
    norb=norb,
    n_reps=1,
    with_final_orbital_rotation=ucj_op.final_orbital_rotation is not None,
)